# Giai đoạn 0 — Chạy thật pipeline VQA tiếng Việt trên LeRobot

> Notebook đồng hành với `vlai-docs/` (đặc biệt **Bài 10** — language/recipes/VQA).
> Toàn bộ Phần 1–4 **chạy thuần CPU, không tải model** — chỉ cần cài được `lerobot` (working tree của repo này).
> Phần 5 là tùy chọn (cần GPU + tải VLM) và được bọc an toàn.

**Mục tiêu:** tự tay dựng dữ liệu VQA tiếng Việt → recipe → render thành `messages` mà policy VLM tiêu thụ.
Đây chính là mắt xích trung tâm của dự án; nắm được nó là bạn nắm được cách "đổ" dữ liệu tiếng Việt vào một VLA/VLM.

Các ô code dưới đây đã được **kiểm chứng chạy được** trên source của repo này.

## 0. Đảm bảo import đúng `lerobot` của working tree

Repo có thể có một bản `lerobot` cũ hơn đã cài trong env. Ô dưới ưu tiên import `lerobot` từ
**source của repo này** (`src/lerobot`). Nếu bạn đã `pip install -e .` thì cũng không sao.

In [ ]:
import sys, pathlib

# Tìm 'src/lerobot/configs/recipe.py' đi lên từ thư mục hiện tại để import đúng working tree.
here = pathlib.Path.cwd()
for p in [here, *here.parents]:
    if (p / "src" / "lerobot" / "configs" / "recipe.py").exists():
        sys.path.insert(0, str(p / "src"))
        print("Dùng lerobot source tại:", p / "src")
        break
else:
    print("Không tìm thấy src/lerobot trong cây thư mục — dùng lerobot đã cài (cần >= bản có configs.recipe).")

import lerobot
print("lerobot:", lerobot.__file__)

## 1. Một mẫu VQA tiếng Việt = cặp 2 event row cùng frame, cùng camera

Theo `datasets/language.py`: style `vqa` là **event-only** và **view-dependent** → mỗi row `vqa`
BẮT BUỘC có `camera` trỏ tới một feature `observation.images.*`. Một mẫu gồm một row `user`
(câu hỏi) và một row `assistant` (câu trả lời), cùng timestamp, cùng camera.

In [ ]:
CAM = "observation.images.top"

# language_events tại một frame (tiếng Việt)
events = [
    {
        "role": "user",
        "style": "vqa",
        "camera": CAM,
        "content": "Vật màu đỏ đang ở bên nào của cái bát?"},
    {   
        "role": "assistant",
        "style": "vqa",
        "camera": CAM,
        "content": "Bên trái cái bát."
    },
]
events

## 2. Recipe VQA tiếng Việt (bindings + messages)

Recipe khai báo **lấy row nào** (`bindings`, dùng resolver `emitted_at(...)`) và **dựng chat turn nào**
(`messages`). Lưu ý: **mọi turn phải có `stream`** (`high_level`/`low_level`) và recipe phải có ít nhất
một turn `target: true` (phần model học sinh ra).

In [ ]:
from lerobot.configs.recipe import TrainingRecipe

recipe = TrainingRecipe.from_dict({
    "bindings": {
        "vqa_query": f"emitted_at(t, style=vqa, role=user, camera={CAM})",
        "vqa":       f"emitted_at(t, style=vqa, role=assistant, camera={CAM})",
    },
    "messages": [
        {"role": "user", "stream": "high_level", "if_present": "vqa_query",
         "content": [
             {"type": "image", "feature": CAM},
             {"type": "text",  "text": "${vqa_query}"},
         ]},
        {"role": "assistant", "stream": "high_level", "content": "${vqa}",
         "target": True, "if_present": "vqa"},
    ],
})
print("Recipe OK | n_messages:", len(recipe.messages), "| bindings:", list(recipe.bindings))

## 3. Render → `messages` mà VLM tiêu thụ

`render_sample(...)` (trong `datasets/language_render.py`) là thứ mà `RenderMessagesStep`
gọi lúc training. Nó trả về 3 thứ:
- `messages` — chat HF-style (đa phương thức: image block + text block),
- `message_streams` — nhãn high/low level để lọc,
- `target_message_indices` — turn nào tính loss (ở đây là câu trả lời, index 1).

In [ ]:
import json
from lerobot.datasets.language_render import render_sample

rendered = render_sample(
    recipe=recipe,
    persistent=[],
    events=events,
    t=0.0,                # timestamp của frame
    sample_idx=0,
    task="Trả lời câu hỏi về ảnh",
)
print(json.dumps(rendered, ensure_ascii=False, indent=2))

**Đọc kết quả:** `target_message_indices = [1]` nghĩa là turn assistant ("Bên trái cái bát.")
là mục tiêu huấn luyện; câu hỏi + ảnh là context. Đây đúng là định dạng VQA chuẩn.

## 4. (Tùy chọn A) Nạp recipe từ file YAML

Trong dự án thật bạn sẽ giữ recipe ở file `.yaml` (ví dụ `recipes/vqa_vi.yaml`) thay vì hard-code.

In [ ]:
import tempfile, os
yaml_text = """
bindings:
  vqa_query: "emitted_at(t, style=vqa, role=user, camera=observation.images.top)"
  vqa:       "emitted_at(t, style=vqa, role=assistant, camera=observation.images.top)"
messages:
  - role: user
    stream: high_level
    if_present: vqa_query
    content:
      - { type: image, feature: observation.images.top }
      - { type: text,  text: "${vqa_query}" }
  - role: assistant
    stream: high_level
    content: "${vqa}"
    target: true
    if_present: vqa
"""
f = tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False)
f.write(yaml_text); f.close()
recipe_yaml = TrainingRecipe.from_yaml(f.name)
os.unlink(f.name)
print("YAML recipe OK | messages:", len(recipe_yaml.messages))

## 4. (Tùy chọn B) Blend: trộn VQA + task-following trong một dataset

`blend` cho phép một dataset huấn luyện **nhiều mục tiêu** cùng lúc. Mỗi sample được chọn một
nhánh **xác định theo `sample_idx`** (hash), nên tỉ lệ tiệm cận trọng số đã khai (ở đây ~70% VQA / 30% task).

In [ ]:
from collections import Counter

blend = TrainingRecipe.from_dict({
  "blend": {
    "vqa": {"weight": 0.7,
      "bindings": {
        "vqa_query": f"emitted_at(t, style=vqa, role=user, camera={CAM})",
        "vqa":       f"emitted_at(t, style=vqa, role=assistant, camera={CAM})"},
      "messages": [
        {"role": "user", "stream": "high_level", "if_present": "vqa_query",
         "content": [{"type": "image", "feature": CAM}, {"type": "text", "text": "${vqa_query}"}]},
        {"role": "assistant", "stream": "high_level", "content": "${vqa}", "target": True, "if_present": "vqa"}]},
    "task_follow": {"weight": 0.3,
      "messages": [
        {"role": "user", "stream": "high_level",
         "content": [{"type": "image", "feature": CAM}, {"type": "text", "text": "${task}"}]},
        {"role": "assistant", "stream": "high_level", "content": "OK.", "target": True}]},
  }
})

def branch_of(r):
    txt = []
    for m in r["messages"]:
        ct = m.get("content")
        if isinstance(ct, list):
            txt += [b.get("text", "") for b in ct if isinstance(b, dict)]
        elif isinstance(ct, str):
            txt.append(ct)
    return "vqa" if any("?" in t for t in txt) else "task_follow"

c = Counter()
for i in range(200):
    r = render_sample(recipe=blend, persistent=[], events=events, t=0.0, sample_idx=i,
                      task="Nhặt khối đỏ bỏ vào bát")
    c[branch_of(r)] += 1
print("Tỉ lệ nhánh trên 200 sample:", dict(c))

## 5. Backbone VLM cho tiếng Việt — soi `SmolVLAConfig`

Chưa tải model — chỉ xem config để biết các nút bấm quan trọng (Bài 09): `vlm_model_name`
(đổi sang VLM đa ngôn ngữ), `tokenizer_max_length` (tăng cho câu tiếng Việt dài).

In [ ]:
try:
    from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
    cfg = SmolVLAConfig()
    print("backbone VLM     :", cfg.vlm_model_name)
    print("tokenizer_max_len:", cfg.tokenizer_max_length, "  # tăng nếu câu hỏi/đáp tiếng Việt dài")
    print("chunk_size       :", cfg.chunk_size)
    print("train_expert_only:", cfg.train_expert_only, "  # cho VQA bạn sẽ cần mở phần ngôn ngữ")
except Exception as e:
    print("Không tải được SmolVLAConfig (có thể thiếu extra). Lỗi:", type(e).__name__, e)
    print("Cài: uv sync --extra smolvla   (hoặc pip install -e '.[smolvla]')")

## 6. Bước tiếp theo (ngoài notebook này)

Bạn vừa chạy thật **mắt xích ngôn ngữ** của pipeline VQA. Để khép kín Giai đoạn 0 → 2:

1. **Dataset** (Bài 12): tạo `LeRobotDataset` có ảnh + ghi cặp VQA tiếng Việt vào `language_events`.
   Dùng `LeRobotDataset.create(repo_id, fps, features, ...)` → `add_frame({...})` → `save_episode()` → `finalize()`.
2. **Tokenizer** (Bài 04): `messages` ở trên được `TokenizerProcessorStep` biến thành
   `OBS_LANGUAGE_TOKENS` cho VLM. Đây là nơi chất lượng **tokenizer tiếng Việt** của backbone quyết định nhiều.
3. **Train/Finetune** (Bài 06, 14): `lerobot-train --policy.path=lerobot/smolvla_base --dataset.repo_id=... --peft.method_type=LORA`.
4. **Model riêng** (Bài 13): đóng gói policy `my_vqa` với backbone đa ngôn ngữ + LM head sinh text.

> Lưu ý trung thực: Phần 1–5 đã được kiểm chứng chạy. Các bước 1–4 ở trên cần GPU/tải model
> và tùy version API — đối chiếu chữ ký hàm trong source trước khi chạy thật.